# Stage 3 error analysis and final selection
The official test set is intentionally absent from this notebook. Finalists are evaluated only on the unchanged validation partition. A conservative threshold for the tuned Random Forest is selected from grouped out-of-fold fit predictions, then assessed once on validation.

In [ ]:
from pathlib import Path
import json, time
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, log_loss
from sklearn.calibration import calibration_curve
from sklearn.utils.class_weight import compute_sample_weight

ROOT=Path('..').resolve(); DATA=ROOT/'data'/'processed'; RESULTS=ROOT/'results'; MODELS=ROOT/'models'; PLOTS=RESULTS/'plots'
SEED, THREADS=42, 2; CLASSES=['Normal','Analysis','Backdoor','DoS','Exploits','Fuzzers']
for filename in ['stage3_finalist_comparison.csv','stage3_per_class_metrics.csv','stage3_error_analysis.csv','stage3_oof_threshold_selection.csv','stage3_model_selection_report.md']:
    assert not (RESULTS/filename).exists(), f'Existing {filename} requires review before rerun.'
frame=pd.read_csv(DATA/'selected_training.csv'); fit_idx=pd.read_csv(DATA/'fit_indices.csv')['row_index'].to_numpy(); val_idx=pd.read_csv(DATA/'validation_indices.csv')['row_index'].to_numpy()
assert len(set(fit_idx)&set(val_idx))==0 and set(fit_idx)|set(val_idx)==set(range(len(frame)))
predictors=[c for c in frame if c not in {'id','label','attack_cat','_fingerprint'}]
assert {'id','label','attack_cat','_fingerprint'}.isdisjoint(predictors)
X_fit,X_val=frame.iloc[fit_idx][predictors],frame.iloc[val_idx][predictors]; y_fit,y_val=frame.iloc[fit_idx].attack_cat,frame.iloc[val_idx].attack_cat
numeric=X_fit.select_dtypes(include='number').columns.tolist(); categorical=X_fit.select_dtypes(exclude='number').columns.tolist()
PLOTS.mkdir(parents=True,exist_ok=True); MODELS.mkdir(parents=True,exist_ok=True)
print('fit/validation',X_fit.shape,X_val.shape,'predictors',len(predictors))

In [ ]:
def make_preprocessor(dense=False):
    return ColumnTransformer([
      ('numeric',SimpleImputer(strategy='median'),numeric),
      ('categorical',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',OneHotEncoder(handle_unknown='ignore',sparse_output=not dense,dtype=np.float32))]),categorical),
    ],sparse_threshold=0 if dense else 1)

def original_rf():
    return Pipeline([('preprocessor',make_preprocessor()),('classifier',RandomForestClassifier(n_estimators=100,random_state=SEED,n_jobs=THREADS))])
def hgb_tuned():
    return Pipeline([('preprocessor',make_preprocessor(dense=True)),('classifier',HistGradientBoostingClassifier(random_state=SEED,max_iter=200,learning_rate=0.06,max_leaf_nodes=15,min_samples_leaf=30,l2_regularization=0.5))])
def aligned_proba(pipeline, X):
    raw=pipeline.predict_proba(X); output=np.zeros((len(X),len(CLASSES)))
    for source_index,label in enumerate(pipeline.classes_): output[:,CLASSES.index(label)]=raw[:,source_index]
    return output
def conservative_labels(proba, threshold):
    labels=np.array(CLASSES)[proba.argmax(axis=1)].astype(object)
    labels[(labels!='Normal') & (proba.max(axis=1)<threshold)]='Normal'
    return labels
def normal_fpr(y,pred):
    normal=y.to_numpy()=='Normal'; return float((pred[normal]!='Normal').mean())
def attack_recall(y,pred):
    attack=y.to_numpy()!='Normal'; return float((pred[attack]!='Normal').mean())
def ece(proba,y):
    confidence=proba.max(axis=1); correct=np.array(CLASSES)[proba.argmax(axis=1)]==y.to_numpy(); bins=np.linspace(0,1,11); total=0.0
    for low,high in zip(bins[:-1],bins[1:]):
        mask=(confidence>=low)&((confidence<high) if high<1 else (confidence<=high))
        if mask.any(): total+=mask.mean()*abs(correct[mask].mean()-confidence[mask].mean())
    return float(total)
def plot_confusions(name, matrix):
    normalized=matrix/matrix.sum(axis=1,keepdims=True)
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    for ax,data,title,fmt in zip(axes,[matrix,normalized],['Raw','Row-normalized'],['d','.2f']):
        ax.imshow(data,cmap='Blues'); ax.set(xticks=range(6),yticks=range(6),xticklabels=CLASSES,yticklabels=CLASSES,xlabel='Predicted',ylabel='Actual',title=f'{name} {title}')
        plt.setp(ax.get_xticklabels(),rotation=35,ha='right')
        for i in range(6):
            for j in range(6): ax.text(j,i,format(data[i,j],fmt),ha='center',va='center',fontsize=7)
    fig.tight_layout(); fig.savefig(PLOTS/f'stage3_{name.lower().replace(' ','_')}_confusions.png',dpi=140); plt.close(fig)

def evaluate(name,pipeline,pred,proba,fit_seconds,phase,threshold=None):
    report=classification_report(y_val,pred,labels=CLASSES,output_dict=True,zero_division=0); matrix=confusion_matrix(y_val,pred,labels=CLASSES)
    probability_order=sorted(CLASSES); ordered_proba=proba[:,[CLASSES.index(label) for label in probability_order]]
    row={'model':name,'phase':phase,'threshold':threshold,'fit_seconds':fit_seconds,'inference_ms_per_row':None,'accuracy':accuracy_score(y_val,pred),'macro_f1':f1_score(y_val,pred,average='macro'),'weighted_f1':f1_score(y_val,pred,average='weighted'),'attack_recall':attack_recall(y_val,pred),'normal_false_positive_rate':normal_fpr(y_val,pred),'log_loss':log_loss(y_val,ordered_proba,labels=probability_order),'expected_calibration_error':ece(proba,y_val),'confusion_matrix':json.dumps(matrix.tolist())}
    for label in CLASSES:
        for metric in ['precision','recall','f1-score']: row[f'{label}_{metric}']=report[label][metric]
    started=time.perf_counter(); pipeline.predict(X_val); row['inference_ms_per_row']=1000*(time.perf_counter()-started)/len(X_val)
    plot_confusions(name,matrix); return row,report,matrix

## Finalists
The original Random Forest uses the Stage 1 configuration. The tuned Random Forest is loaded from Stage 2. HistGradientBoosting uses the selected Stage 2 configuration with fit-only balanced sample weights.

In [ ]:
finalists={}
start=time.perf_counter(); original=original_rf().fit(X_fit,y_fit); finalists['Original Random Forest']=(original,time.perf_counter()-start)
tuned=joblib.load(MODELS/'stage2_best_validation_pipeline.joblib'); finalists['Tuned Random Forest']=(tuned,0.0)
start=time.perf_counter(); hgb=hgb_tuned().fit(X_fit,y_fit,classifier__sample_weight=compute_sample_weight('balanced',y_fit)); finalists['Tuned HistGradientBoosting']=(hgb,time.perf_counter()-start)
raw_results=[]; artifacts={}
for name,(pipe,seconds) in finalists.items():
    proba=aligned_proba(pipe,X_val); pred=np.array(CLASSES)[proba.argmax(axis=1)]
    row,report,matrix=evaluate(name,pipe,pred,proba,seconds,'raw')
    raw_results.append(row); artifacts[name]={'pipeline':pipe,'proba':proba,'pred':pred,'report':report,'matrix':matrix}
    print(name,round(row['macro_f1'],4),round(row['normal_false_positive_rate'],4),round(row['fit_seconds'],1),flush=True)

## Fit-only out-of-fold conservative threshold
A twofold fingerprint-grouped split of fit rows selects a confidence threshold. It can only convert a predicted attack to Normal, which targets false alarms. Validation is used once to assess the selected threshold.

In [ ]:
fit_groups=pd.util.hash_pandas_object(X_fit,index=False,categorize=True).astype('uint64'); cv=list(StratifiedGroupKFold(n_splits=2,shuffle=True,random_state=SEED).split(X_fit,y_fit,fit_groups))
assert all(set(fit_groups.iloc[a]).isdisjoint(set(fit_groups.iloc[b])) for a,b in cv)
oof=np.zeros((len(X_fit),len(CLASSES)))
for train_fold,holdout_fold in cv:
    pipe=Pipeline([('preprocessor',make_preprocessor()),('classifier',RandomForestClassifier(n_estimators=50,max_depth=22,min_samples_leaf=2,class_weight='balanced_subsample',random_state=SEED,n_jobs=THREADS))])
    pipe.fit(X_fit.iloc[train_fold],y_fit.iloc[train_fold]); oof[holdout_fold]=aligned_proba(pipe,X_fit.iloc[holdout_fold])
threshold_rows=[]
for threshold in np.arange(0.50,0.96,0.05):
    pred=conservative_labels(oof,threshold); threshold_rows.append({'threshold':round(float(threshold),2),'oof_macro_f1':f1_score(y_fit,pred,average='macro'),'oof_accuracy':accuracy_score(y_fit,pred),'oof_attack_recall':attack_recall(y_fit,pred),'oof_normal_false_positive_rate':normal_fpr(y_fit,pred)})
thresholds=pd.DataFrame(threshold_rows); feasible=thresholds[thresholds.oof_normal_false_positive_rate<=0.10]
selected_threshold=(feasible.sort_values(['oof_macro_f1','oof_attack_recall'],ascending=False).iloc[0] if len(feasible) else thresholds.sort_values(['oof_normal_false_positive_rate','oof_macro_f1'],ascending=[True,False]).iloc[0])
thresholds['selected']=thresholds.threshold.eq(selected_threshold.threshold); thresholds.to_csv(RESULTS/'stage3_oof_threshold_selection.csv',index=False)
adjusted_pred=conservative_labels(artifacts['Tuned Random Forest']['proba'],selected_threshold.threshold)
adjusted_row,adjusted_report,adjusted_matrix=evaluate('Tuned Random Forest conservative',tuned,adjusted_pred,artifacts['Tuned Random Forest']['proba'],0.0,'oof_threshold',float(selected_threshold.threshold))
raw_results.append(adjusted_row); artifacts['Tuned Random Forest conservative']={'pipeline':tuned,'proba':artifacts['Tuned Random Forest']['proba'],'pred':adjusted_pred,'report':adjusted_report,'matrix':adjusted_matrix}
print('selected threshold',selected_threshold.to_dict()); print('validation',round(adjusted_row['macro_f1'],4),round(adjusted_row['normal_false_positive_rate'],4),flush=True)

In [ ]:
comparison=pd.DataFrame(raw_results)
eligible=comparison[comparison.normal_false_positive_rate<=0.10]
selected=(eligible.sort_values(['macro_f1','attack_recall'],ascending=False).iloc[0] if len(eligible) else comparison.sort_values(['normal_false_positive_rate','macro_f1'],ascending=[True,False]).iloc[0])
selection_reason='Highest validation macro-F1 among candidates meeting the 10% normal-FPR operating target' if len(eligible) else 'Lowest normal FPR operating point'
comparison['selection']='REVIEW'; comparison.loc[comparison.model.eq(selected.model),'selection']='SELECTED'
comparison['selection_reason']='Validation comparison retained for error analysis'
comparison.loc[comparison.model.eq(selected.model),'selection_reason']=selection_reason
comparison.sort_values('macro_f1',ascending=False).to_csv(RESULTS/'stage3_finalist_comparison.csv',index=False)
per_class=[]
for name,item in artifacts.items():
    for label in CLASSES:
        values=item['report'][label]; per_class.append({'model':name,'class':label,'precision':values['precision'],'recall':values['recall'],'f1':values['f1-score'],'support':int(values['support'])})
per_class=pd.DataFrame(per_class); per_class.to_csv(RESULTS/'stage3_per_class_metrics.csv',index=False)
fig,ax=plt.subplots(figsize=(11,4)); pivot=per_class[per_class['class'].isin(['Analysis','Backdoor','DoS'])].pivot(index='model',columns='class',values='recall'); pivot.plot.bar(ax=ax,rot=25); ax.set_ylim(0,1); ax.set_ylabel('Recall'); fig.tight_layout(); fig.savefig(PLOTS/'stage3_minority_recall_comparison.png',dpi=140); plt.close(fig)
fig,ax=plt.subplots(figsize=(10,4)); comparison.set_index('model')[['macro_f1','attack_recall','normal_false_positive_rate']].plot.bar(ax=ax,rot=20); ax.set_ylim(0,1); fig.tight_layout(); fig.savefig(PLOTS/'stage3_finalist_tradeoffs.png',dpi=140); plt.close(fig)
display(comparison.sort_values('macro_f1',ascending=False)[['model','phase','threshold','macro_f1','accuracy','Analysis_recall','Backdoor_recall','DoS_recall','attack_recall','normal_false_positive_rate','inference_ms_per_row','selection']])

In [ ]:
selected_name=selected.model; selected_item=artifacts[selected_name]; selected_pred=selected_item['pred']; selected_matrix=selected_item['matrix']
all_fingerprints=pd.util.hash_pandas_object(frame[predictors],index=False,categorize=True); conflict_groups=all_fingerprints.groupby(all_fingerprints).transform('size').index if False else None
group_labels=frame.groupby(all_fingerprints)['attack_cat'].nunique(); validation_conflict=all_fingerprints.iloc[val_idx].map(group_labels).gt(1).to_numpy()
errors=[]
for label in ['Analysis','Backdoor','DoS']:
    row_index=CLASSES.index(label); row=selected_matrix[row_index]; wrong=row.copy(); wrong[row_index]=0; main=CLASSES[int(wrong.argmax())]
    errors.append({'analysis':'minority_error','class':label,'support':int(row.sum()),'recall':float(row[row_index]/row.sum()),'most_frequent_confusion':main,'confused_count':int(wrong.max()),'conflict_group_rows':int(((y_val.to_numpy()==label)&validation_conflict).sum())})
for actual_i,actual in enumerate(CLASSES):
    for pred_i,predicted in enumerate(CLASSES):
        if actual_i!=pred_i and selected_matrix[actual_i,pred_i]: errors.append({'analysis':'confusion_pair','class':actual,'predicted_as':predicted,'count':int(selected_matrix[actual_i,pred_i])})
errors.append({'analysis':'conflicting_predictor_groups','validation_rows_in_conflicting_groups':int(validation_conflict.sum()),'validation_error_rate_conflicting':float((selected_pred[validation_conflict]!=y_val.to_numpy()[validation_conflict]).mean()),'validation_error_rate_other':float((selected_pred[~validation_conflict]!=y_val.to_numpy()[~validation_conflict]).mean())})
errors.append({'analysis':'class_imbalance','fit_class_counts':json.dumps(y_fit.value_counts().to_dict()),'validation_class_counts':json.dumps(y_val.value_counts().to_dict())})
pd.DataFrame(errors).to_csv(RESULTS/'stage3_error_analysis.csv',index=False)
fig,ax=plt.subplots(figsize=(7,5))
for name,item in artifacts.items():
    attack_true=y_val.to_numpy()!='Normal'; fraction,mean=calibration_curve(attack_true,1-item['proba'][:,CLASSES.index('Normal')],n_bins=10,strategy='quantile'); ax.plot(mean,fraction,marker='o',label=name)
ax.plot([0,1],[0,1],'k--'); ax.set(xlabel='Mean predicted attack probability',ylabel='Observed attack rate',title='Attack probability calibration'); ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(PLOTS/'stage3_attack_calibration.png',dpi=140); plt.close(fig)
metadata={'selected_model':selected_name,'selection_rule':selection_reason,'validation_macro_f1':float(selected.macro_f1),'validation_accuracy':float(selected.accuracy),'normal_false_positive_rate':float(selected.normal_false_positive_rate),'attack_recall':float(selected.attack_recall),'per_class_recall':{label:float(selected[f'{label}_recall']) for label in CLASSES},'threshold':None if pd.isna(selected.threshold) else float(selected.threshold),'feature_list':predictors,'class_labels':CLASSES,'fit_rows':len(X_fit),'validation_rows':len(X_val),'official_test_used':False}
joblib.dump(selected_item['pipeline'],MODELS/'stage3_selected_pipeline.joblib'); (MODELS/'stage3_selected_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
report=f'''# Stage 3 model selection

Selected model: {selected_name}

Validation macro-F1: {selected.macro_f1:.4f}

Validation accuracy: {selected.accuracy:.4f}

Attack recall: {selected.attack_recall:.4f}

Normal false-positive rate: {selected.normal_false_positive_rate:.4f}

Selection rule: {selection_reason}

Minority-class recalls: Analysis {selected.Analysis_recall:.4f}, Backdoor {selected.Backdoor_recall:.4f}, DoS {selected.DoS_recall:.4f}.

The threshold, where present, was selected from grouped out-of-fold predictions on fit rows only. The official test set was not loaded or evaluated.
'''
(RESULTS/'stage3_model_selection_report.md').write_text(report,encoding='utf-8')
assert (MODELS/'stage3_selected_pipeline.joblib').is_file() and (MODELS/'stage3_selected_metadata.json').is_file()
display(pd.DataFrame(errors).head(20))